In [6]:
#!/usr/bin/env Rscript

requireNamespace("anndata", quietly=TRUE)
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(purrr)
  library(tibble)
  library(edgeR)
  library(limma)
  library(Matrix)
})

# Start timer
start_time <- Sys.time()

## VIASH START
par <- list(
  input      = "../../data/processed/sciplex/pseudbulked_sciplex3.h5ad",
  output_dir = "../../data/processed/sciplex/deg",   # make sure this exists
  control    = "control",
  # Testing parameters - set to NULL to run full analysis
  test_mode  = TRUE,           # Enable test mode
  max_cell_lines = 3,          # Process only first N cell lines
  max_perturbations = 3,       # Process only first N perturbations per cell line
  max_genes = 1000,            # Use only top N variable genes
  specific_times = c(24),      # Analyze only specific time points (NULL for all)
  specific_perturbations = NULL # Analyze only these perturbations (e.g., c("Drug1", "Drug2"))
)

# Example test configurations:
# 1. Quick test (current settings) - ~1-2 minutes
# 2. Single drug test:
#    specific_perturbations = c("Belinostat"), max_cell_lines = 1
# 3. Time course test:
#    specific_times = c(8, 24, 72), max_perturbations = 2
# 4. Full analysis:
#    test_mode = FALSE (or set all limits to NULL)

## VIASH END

# helper to sanitize names for model.matrix/contrasts
clean <- function(x) gsub("[^[:alnum:]_]", "_", x)

# Define which DE results to store in layers
res_cols <- c("logFC", "AveExpr", "t", "P.Value", "adj.P.Value", "B")

# load the full pseudobulk AnnData
adata <- anndata::read_h5ad(par$input)

# Test mode subsetting
if (!is.null(par$test_mode) && par$test_mode) {
  message("\n⚡ TEST MODE ENABLED ⚡")
  message("Original dataset dimensions: ", nrow(adata$obs), " samples x ", ncol(adata$obs), " genes")
  
  # Subset cell lines
  cell_lines <- unique(adata$obs$cell_line)
  if (!is.null(par$max_cell_lines) && length(cell_lines) > par$max_cell_lines) {
    cell_lines <- cell_lines[1:par$max_cell_lines]
    message("Subsetting to ", par$max_cell_lines, " cell lines: ", paste(cell_lines, collapse=", "))
    adata <- adata[adata$obs$cell_line %in% cell_lines, ]
  }
  
  # Subset time points
  if (!is.null(par$specific_times)) {
    message("Filtering to time points: ", paste(par$specific_times, collapse=", "), " hours")
    adata <- adata[adata$obs$time %in% par$specific_times, ]
  }
  
  # Subset perturbations 
  if (!is.null(par$specific_perturbations)) {
    # Use specific perturbations if provided
    keep_perts <- c(par$control, par$specific_perturbations)
    message("Using specific perturbations: ", paste(keep_perts, collapse=", "))
    adata <- adata[adata$obs$perturbation %in% keep_perts, ]
  } else if (!is.null(par$max_perturbations)) {
    # Otherwise limit to max_perturbations
    perturbations <- unique(adata$obs$perturbation)
    treated_perts <- setdiff(perturbations, par$control)
    if (length(treated_perts) > par$max_perturbations) {
      keep_perts <- c(par$control, treated_perts[1:par$max_perturbations])
      message("Subsetting to ", par$max_perturbations, " perturbations (+ control): ", 
              paste(keep_perts, collapse=", "))
      adata <- adata[adata$obs$perturbation %in% keep_perts, ]
    }
  }
  
  # Subset genes - keep most variable
  if (!is.null(par$max_genes) && ncol(adata) > par$max_genes) {
    message("Calculating gene variance for subsetting...")
    gene_vars <- apply(as.matrix(adata$X), 2, var)
    top_genes <- order(gene_vars, decreasing = TRUE)[1:par$max_genes]
    adata <- adata[, top_genes]
    message("Subset to top ", par$max_genes, " most variable genes")
  }
  
  message("Test mode dataset: ", nrow(adata$obs), " samples x ", ncol(adata), " genes")
  
  # Show analysis overview
  analysis_summary <- adata$obs %>%
    group_by(cell_line, perturbation, time) %>%
    summarise(n_samples = n(), .groups = "drop") %>%
    arrange(cell_line, time, perturbation)
  
  message("\nConditions to analyze:")
  print(analysis_summary)
  message("")
}

cell_lines_to_process <- unique(adata$obs$cell_line)
n_cell_lines <- length(cell_lines_to_process)
cl_num <- 0

for (cl in cell_lines_to_process) {
  cl_num <- cl_num + 1
  cl_start <- Sys.time()
  message("\n▶︎ Processing cell_line ", cl_num, "/", n_cell_lines, ": ", cl)
  
  # subset to this cell_line
  ad  <- adata[adata$obs$cell_line == cl, ]
  obs <- ad$obs %>%
    mutate(
        perturbation = factor(perturbation),
        dose_value   = as.numeric(dose_value),
        time         = as.numeric(time),
        # collapsed + cleaned conditions
        raw_cond = if_else(
          perturbation == par$control,
          paste0(par$control, "_", time, "h"),
          paste0(perturbation, "_", dose_value, "nM_", time, "h")
        ),
        cond = factor(clean(raw_cond)),  # sanitize values here!
        plate = factor(plate)
    )
  
  # Check for required controls
  control_times <- obs %>%
    filter(perturbation == par$control) %>%
    pull(time) %>%
    unique()
  
  treated_times <- obs %>%
    filter(perturbation != par$control) %>%
    pull(time) %>%
    unique()
  
  missing_controls <- setdiff(treated_times, control_times)
  if (length(missing_controls) > 0) {
    warning("Missing controls for time points: ", paste(missing_controls, collapse=", "))
  }
  
  # build DGEList + design
  counts <- Matrix::t(ad$X)
  dge    <- DGEList(counts=counts)
  design <- model.matrix(
    ~ 0 + cond + plate,
    data = obs
  )
  
  # filter genes and normalize
  keep <- filterByExpr(dge, design)
  dge  <- dge[keep, , keep.lib.sizes=FALSE] %>% calcNormFactors()
  
  # voom + lmFit
  v   <- voom(dge, design, plot=FALSE)
  fit <- lmFit(v, design)
  
  # pick out only the treated conds (we won't DE on control-vs-control)
  new_obs <- obs %>%
    distinct(cond, perturbation, dose_value, time, raw_cond) %>%
    filter(perturbation != par$control)
  
  # run one contrast per treated cond vs. the matching control@same time
  n_contrasts <- nrow(new_obs)
  contrast_num <- 0
  
  de_res <- pmap_dfr(new_obs, function(cond, perturbation, dose_value, time, raw_cond, ...) {
    contrast_num <<- contrast_num + 1
    if (contrast_num %% 5 == 1 || contrast_num == n_contrasts) {
      message(sprintf("    Running contrast %d/%d", contrast_num, n_contrasts))
    }
    
    raw_control <- paste0(par$control, "_", time, "h")
    
    # Check if control exists for this time point
    if (!any(obs$raw_cond == raw_control)) {
      warning("No control found for time ", time, "h, skipping ", raw_cond)
      return(NULL)
    }
    
    contrast <- paste0("cond", clean(raw_cond), " - cond", clean(raw_control))
    
    tryCatch({
      ctr <- makeContrasts(contrasts = contrast, levels = colnames(coef(fit)))
      fit2 <- contrasts.fit(fit, ctr) %>% 
        eBayes(robust = !isTRUE(par$test_mode))  # Skip robust for speed in test mode
      
      topTable(fit2, number = Inf, sort = "none") %>%
        rownames_to_column("gene") %>%
        mutate(
          cond = clean(raw_cond),
          perturbation = perturbation,
          dose_value = dose_value,
          time = time
        )
    }, error = function(e) {
      warning("Error in contrast for ", raw_cond, ": ", e$message)
      return(NULL)
    })
  })
  
  # Skip if no valid DE results
  if (is.null(de_res) || nrow(de_res) == 0) {
    warning("No valid DE results for cell line ", cl)
    next
  }
  
  # adjust p-values globally across all contrasts
  de_df <- de_res %>%
    mutate(
      adj.P.Value = p.adjust(P.Value, method="BH")
    )
  
  obs_out <- new_obs %>%
    filter(clean(raw_cond) %in% unique(de_df$cond)) %>%  # only conds with results
    arrange(cond) %>%                                     # sort by cond
    remove_rownames() %>%                                 # DUMP any existing row.names
    column_to_rownames("cond") %>%                        # now safe to turn "cond" into rownames
    select(perturbation, dose_value, time)
  
  genes   <- unique(de_df$gene)
  var_out <- data.frame(gene = genes, row.names = genes)
  
  # Create layers for each DE statistic
  layers <- map(res_cols, function(m) {
    if (!m %in% names(de_df)) {
      warning("Column ", m, " not found in DE results")
      return(NULL)
    }
    
    de_df %>%
      select(gene, cond, !!sym(m)) %>%
      pivot_wider(names_from = gene, values_from = !!sym(m)) %>%
      arrange(match(cond, rownames(obs_out))) %>%
      select(-cond) %>%
      as.matrix()
  }) %>% 
    set_names(res_cols) %>%
    compact()  # Remove NULL entries
  
  # carry over global uns if you like
  new_uns <- adata$uns
  
  # assemble and write
  out_adata <- anndata::AnnData(
    obs    = obs_out,
    var    = var_out,
    layers = layers,
    uns    = new_uns
  )
  
  # Create output directory if it doesn't exist
  if (!dir.exists(par$output_dir)) {
    dir.create(par$output_dir, recursive = TRUE)
  }
  
  outfile <- file.path(par$output_dir, paste0(cl, "_de.h5ad"))
  message("  Writing: ", outfile)
  out_adata$write_h5ad(outfile, compression = "gzip")
  
  # Show time for this cell line
  cl_time <- difftime(Sys.time(), cl_start, units = "secs")
  message(sprintf("  ✓ Cell line completed in %.1f seconds", cl_time))
}

message("DE analysis complete!")

# Show runtime
end_time <- Sys.time()
runtime <- difftime(end_time, start_time, units = "secs")
message(sprintf("\nTotal runtime: %.1f seconds", runtime))

if (!is.null(par$test_mode) && par$test_mode) {
  message("\n📝 Note: This was a TEST RUN with reduced data.")
  message("Set test_mode = FALSE for full analysis.")
}


<U+26A1> TEST MODE ENABLED <U+26A1>

Original dataset dimensions: 4748 samples x 17 genes

Filtering to time points: 24 hours

Subsetting to 3 perturbations (+ control): control, MK-0752, Pelitinib (EKB-569), SL-327

Calculating gene variance for subsetting...

Subset to top 1000 most variable genes

Test mode dataset: 166 samples x 1000 genes


Conditions to analyze:



# A tibble: 12 x 4
   cell_line perturbation        time  n_samples
   <fct>     <fct>               <fct>     <int>
 1 A549      MK-0752             24            8
 2 A549      Pelitinib (EKB-569) 24            8
 3 A549      SL-327              24            8
 4 A549      control             24           32
 5 K562      MK-0752             24            8
 6 K562      Pelitinib (EKB-569) 24            7
 7 K562      SL-327              24            8
 8 K562      control             24           32
 9 MCF7      MK-0752             24            8
10 MCF7      Pelitinib (EKB-569) 24            7
11 MCF7      SL-327              24            8
12 MCF7      control             24           32





<U+25B6><U+FE0E> Processing cell_line 1/3: A549

    Running contrast 1/12

    Running contrast 6/12

    Running contrast 11/12

    Running contrast 12/12

  Writing: ../../data/processed/sciplex/deg/A549_de.h5ad

  <U+2713> Cell line completed in 0.3 seconds


<U+25B6><U+FE0E> Processing cell_line 2/3: MCF7

    Running contrast 1/12

    Running contrast 6/12

    Running contrast 11/12

    Running contrast 12/12

  Writing: ../../data/processed/sciplex/deg/MCF7_de.h5ad

  <U+2713> Cell line completed in 0.3 seconds


<U+25B6><U+FE0E> Processing cell_line 3/3: K562

    Running contrast 1/12

    Running contrast 6/12

    Running contrast 11/12

    Running contrast 12/12

  Writing: ../../data/processed/sciplex/deg/K562_de.h5ad

  <U+2713> Cell line completed in 0.3 seconds

DE analysis complete!


Total runtime: 10.1 seconds


<U+0001F4DD> Note: This was a TEST RUN with reduced data.

Set test_mode = FALSE for full analysis.

